# 3. Full Pipeline: Cell Video → Classroom Video
Goal: combine detection + tracking, run it on real footage, then adapt it to people.

## 3.1 Setup
Reuse `detect_objects` (NB1) and `MultiObjectTracker` (NB2). Easiest for now: re-run those notebooks first, or copy the functions into a shared `tracking_utils.py` module.

In [ ]:
%run 01_detection_basics.ipynb

In [ ]:
%run 02_tracking_algorithms.ipynb

## 3.2 Run the pipeline on the cell video

In [ ]:
cap = cv2.VideoCapture("cell_video.avi")
tracker = MultiObjectTracker(max_missed=5, cost_threshold=40)
lower = np.array([0, 175, 0]); upper = np.array([255, 255, 255])

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("cell_tracked.mp4", fourcc, 20,
                      (int(cap.get(3)), int(cap.get(4))))

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    centroids, boxes = detect_objects(frame, lower, upper)
    tracked = tracker.update(centroids)

    for tid, c in tracked.items():
        cv2.circle(frame, (int(c[0]), int(c[1])), 3, (0, 0, 255), -1)
        cv2.putText(frame, f"ID {tid}", (int(c[0]), int(c[1]) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
    out.write(frame)

cap.release()
out.release()
print("done")

## 3.3 Adapting detection for classroom video
Cells are colored blobs — people aren't. We need a different detector.

### Option A — background subtraction (fast, no model)

In [ ]:
backSub = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=25, detectShadows=True)

In [ ]:
def detect_people_bgsub(frame, backSub, area_range=(800, 40000)):
    fg = backSub.apply(frame)
    fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)[1]  # drop shadow pixels (127)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    good = [c for c in contours if area_range[0] < cv2.contourArea(c) < area_range[1]]
    centroids = [get_centroid(c) for c in good]
    boxes = [cv2.boundingRect(c) for c in good]
    return centroids, boxes

Needs a static camera and ~30-50 warm-up frames before it's reliable — fine for a fixed classroom camera.

### Option B — pretrained person detector (more robust)

In [ ]:
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

In [ ]:
def detect_people_hog(frame):
    boxes, weights = hog.detectMultiScale(frame, winStride=(8, 8), padding=(8, 8), scale=1.05)
    centroids = [(x + w / 2, y + h / 2) for (x, y, w, h) in boxes]
    return centroids, boxes

HOG is CPU-friendly but dated. Later you can swap in a YOLOv8 person detector — keep the same return shape (centroids, boxes) and everything below still works.

## 3.4 Full pipeline on classroom video

In [ ]:
cap = cv2.VideoCapture("classroom_video.mp4")
tracker = MultiObjectTracker(max_missed=10, cost_threshold=80)

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("classroom_tracked.mp4", fourcc, 20,
                      (int(cap.get(3)), int(cap.get(4))))

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    centroids, boxes = detect_people_hog(frame)
    tracked = tracker.update(centroids)

    for tid, c in tracked.items():
        cv2.circle(frame, (int(c[0]), int(c[1])), 4, (0, 0, 255), -1)
        cv2.putText(frame, f"ID {tid}", (int(c[0]), int(c[1]) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    out.write(frame)

cap.release()
out.release()
print("done")

## 3.5 Next steps
- Swap HOG for YOLOv8 (`ultralytics`) — much better person detection
- Try DeepSORT for appearance-based re-ID (handles occlusion better than centroid distance)
- Add analytics on top of `tracked`: count per frame, dwell time per ID (like the `analysisResults` table in the MATLAB example)